# Phase 2: Weight of Evidence (WoE) Binning & Information Value (IV) Feature Selection
This notebook implements true credit scorecard feature engineering. We bin numerical and categorical variables, replace raw inputs with their WoE values, and calculate Information Value (IV) to select stable risk features.


In [1]:
import pandas as pd
import os
import sys

# Ensure we are running from the project root directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.append(os.path.abspath('src'))
from data_processing import DataProcessor
from woe_binning import WoEBinning


## 1. Load Cleaned Splits
We load the cleaned splits (Train/Test and Out-of-Time).


In [2]:
processor = DataProcessor('data/loan.csv')
processor.clean_data()
train_df, oot_df = processor.split_data()
print('Train size:', train_df.shape)
print('OOT size:', oot_df.shape)


Train size: (18061, 116)
OOT size: (20516, 116)


## 2. Fit WoE Mappings
We fit automated binning (Decision-Tree based for numeric and class groupings for categorical) on the training set.


In [3]:
numeric_features = [
    'loan_amnt', 'annual_inc', 'dti', 'revol_util', 
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 
    'pub_rec', 'pub_rec_bankruptcies', 'credit_history_age',
    'emp_length'
]
categorical_features = ['home_ownership', 'purpose']

woe_model = WoEBinning(target_col='target')
woe_model.fit_all(train_df, numeric_features, categorical_features)


Fitting Numeric Columns...
  Fitting loan_amnt...
  Fitting annual_inc...
  Fitting dti...
  Fitting revol_util...
  Fitting delinq_2yrs...
  Fitting inq_last_6mths...
  Fitting open_acc...
  Fitting pub_rec...
  Fitting pub_rec_bankruptcies...
  Fitting credit_history_age...
  Fitting emp_length...
Fitting Categorical Columns...
  Fitting home_ownership...
  Fitting purpose...


## 3. Information Value (IV) Ranking & Selection
Rank characteristics based on their predictive power (IV) and filter out useless (<0.02) and suspicious (>0.50) features.


In [4]:
iv_df = pd.read_csv('outputs/scorecards/iv_report.csv') if pd.io.common.file_exists('outputs/scorecards/iv_report.csv') else pd.DataFrame(woe_model.iv_report)
print('IV Feature Rankings:')
print(iv_df)


IV Feature Rankings:
                 feature  information_value             strength
0             revol_util           0.081813   Weak (0.02 - 0.10)
1                purpose           0.060973   Weak (0.02 - 0.10)
2         inq_last_6mths           0.057900   Weak (0.02 - 0.10)
3             annual_inc           0.042731   Weak (0.02 - 0.10)
4                pub_rec           0.030029   Weak (0.02 - 0.10)
5   pub_rec_bankruptcies           0.022562   Weak (0.02 - 0.10)
6              loan_amnt           0.017834  Useless (IV < 0.02)
7                    dti           0.015228  Useless (IV < 0.02)
8               open_acc           0.014260  Useless (IV < 0.02)
9     credit_history_age           0.013427  Useless (IV < 0.02)
10            emp_length           0.013348  Useless (IV < 0.02)
11        home_ownership           0.002938  Useless (IV < 0.02)
12           delinq_2yrs           0.001054  Useless (IV < 0.02)


## 4. Inspecting WoE Lookup Tables
Let's look at the generated Weight of Evidence mapping rules for a key feature like `annual_inc`.


In [5]:
woe_df = pd.read_csv('outputs/scorecards/woe_tables.csv') if pd.io.common.file_exists('outputs/scorecards/woe_tables.csv') else None
if woe_df is not None:
    print(woe_df[woe_df['variable'] == 'annual_inc'])


     variable              bin_name  total_count  good_count  bad_count  \
5  annual_inc           <= 26777.97         1553        1248        305   
6  annual_inc  (26777.97, 36930.63]         2214        1877        337   
7  annual_inc   (36930.63, 60202.0]         6345        5529        816   
8  annual_inc    (60202.0, 66060.0]         1080         923        157   
9  annual_inc             > 66060.0         6869        6113        756   

   default_rate       woe        iv  
5      0.196394 -0.480726  0.023602  
6      0.152213 -0.172365  0.003879  
7      0.128605  0.023636  0.000195  
8      0.145370 -0.118328  0.000874  
9      0.110060  0.200420  0.014181  


## 5. WoE Transformation
We transform the raw training features to WoE values for model input.


In [6]:
train_woe = woe_model.transform(train_df)
print('Transformed WoE DataFrame Head:')
print(train_woe.head())


Transformed WoE DataFrame Head:
       loan_amnt_woe  annual_inc_woe   dti_woe  revol_util_woe  \
21429      -0.128185        0.023636  0.024242       -0.082154   
21452       0.134953        0.200420 -0.114436       -0.082154   
21463       0.134953        0.200420 -0.114436       -0.082154   
21465      -0.091733        0.200420  0.138343       -0.082154   
21467       0.134953        0.200420  0.138343       -0.014071   

       delinq_2yrs_woe  inq_last_6mths_woe  open_acc_woe  pub_rec_woe  \
21429         0.010127           -0.101796     -0.181904     0.045736   
21452         0.010127           -0.054978      0.053927     0.045736   
21463        -0.054467           -0.054978     -0.181904     0.045736   
21465         0.010127            0.223951      0.053927     0.045736   
21467         0.010127           -0.054978      0.053927     0.045736   

       pub_rec_bankruptcies_woe  credit_history_age_woe  emp_length_woe  \
21429                  0.045804                0.101564  